Name: Rafael Chetat
Subject: AI Quiz 4
Date: 11/20

In [2]:
from pyspark.sql import SparkSession
import pandas as pd

# create spark session
spark = (SparkSession.builder.appName("Disease Database").getOrCreate())

25/11/21 15:44:23 WARN Utils: Your hostname, Rs-MacBook-Air-2.local resolves to a loopback address: 127.0.0.1; using 192.168.1.164 instead (on interface en0)
25/11/21 15:44:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


25/11/21 15:44:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
edges = spark.sparkContext.textFile('edges.tsv')
nodes = spark.sparkContext.textFile('nodes.tsv')

In [ ]:
# Q1
# For each drug, compute the number of genes
# and the number of diseases associated with the
# drug. Output results with top 5 number of genes in a
# descending order
from operator import add

def query1(edges):
    genes = edges.filter(lambda x: 'CbG' in x).map(lambda x: (x.split("\t")[0], 1)).reduceByKey(add).collect()
    diseases = edges.filter(lambda x: 'CtD' in x).map(lambda x: (x.split("\t")[0], 1)).reduceByKey(add).collect()
    genes.sort(key=lambda x: x[1], reverse=True)
    top_5_compounds = genes[:5]
    top_5_compounds_named = nodes.filter(lambda x: any(compound[0] in x for compound in top_5_compounds)).map(lambda x: (x.split("\t")[0], x.split("\t")[1])).collectAsMap()
    print(top_5_compounds_named)
    for compound in top_5_compounds:
        print(f"{compound[0]}, Number of Genes: {compound[1]}, Number of Diseases: {dict(diseases).get(compound[0], 0)}\n")

    print("\n")

query1(edges)

In [ ]:
# Q2: Compute the number of diseases associated
# with 1, 2, 3, …, n drugs. Output results with the top
# 5 number of diseases in a descending order.

def query2(edge_rdd):
    diseases = edges.filter(lambda x: 'CtD' in x).map(lambda x: (x.split("\t")[0], 1)).reduceByKey(add).collect()
    diseases.sort(key=lambda x: x[1], reverse=True)
    top_5_disease = diseases[:5]
    for row in top_5_disease:
        print(f"{row[0]} -> {row[1]} diseases\n")

query2(edges)

Compound::DB00563 -> 19 diseases

Compound::DB00997 -> 17 diseases

Compound::DB00635 -> 15 diseases

Compound::DB00445 -> 14 diseases

Compound::DB00860 -> 13 diseases



In [ ]:
#  Q3: Get the name of drugs that have the top 5
# number of genes. Out put the results.

def query3(edge_rdd):
    gene = edges.filter(lambda x: 'CbG' in x).map(lambda x: (x.split("\t")[0], 1)).reduceByKey(add).collect()
    gene.sort(key=lambda x: x[1], reverse=True)
    top_5_compounds = gene[:5]
    for compound in top_5_compounds:
        print(f"{compound[0]} -> {compound[1]} genes\n")

query3(edges)

Compound::DB01268 -> 132 genes

Compound::DB06616 -> 104 genes

Compound::DB08865 -> 85 genes

Compound::DB01254 -> 64 genes

Compound::DB00331 -> 56 genes



In [ ]:
# end session
spark.stop()